In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q scipy
!pip install -q h5py
!pip install -q matplotlib
!pip install -q torch==1.13.1+cu117 torchvision==0.14.1+cu117 torchaudio==0.13.1 --extra-index-url https://download.pytorch.org/whl/cu117
!pip install -q transformers
!pip install -q fuzzy_match
!pip install -q nltk
!pip install -q rouge
!pip install -q diffusers
!pip install -q accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 GB 1.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.3/24.3 MB 18.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 60.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchdata 0.7.0 requires torch==2.1.0, but you have torch 1.13.1+cu117 which is incompatible.
torchtext 0.16.0 requires torch==2.1.0, but you have torch 1.13.1+cu117 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.7/265.7 kB 5.4 MB/s eta 0:00:00


In [1]:
%cd /content/drive/MyDrive/Research/FINAL/Code/MMMM/

/content/drive/MyDrive/Research/FINAL/Code/MMMM


In [2]:
import sys
sys.path.append("./models")
sys.path.append("./training")
sys.path.append("./testing")
sys.path.append("./utils")
sys.path.append("./ZuCo")
sys.path.append("./trainer")

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

torch.set_default_dtype(torch.float32)

from master_init import *
from DSG import *

In [3]:
config = {
    "device" : "cuda:0",
    "device_ids" : [0]
}
device = config["device"]
device_ids = config["device_ids"]

# model = INITIALIZE_MODEL(device=device, device_ids=device_ids).to(device)
model = INITIALIZE_MODEL(device=device, device_ids=device_ids).to(device, dtype=torch.float32)

In [4]:
dataset_dict = INITIALIZE_DATALOADERS(
    keys=["ZuCo-CLIP"],
    bsz=[64],
    dev_bsz=[64]
)

## Code

In [5]:
# import torch
# import torch.nn as nn
# import torch.nn.functional as F

# def train(args_dict):
#     dataloader = args_dict["dataloader"]
#     model = args_dict["model"]
#     optimizer = args_dict["optimizer"]
#     tokenizer = args_dict["tokenizer"]
#     criterion = args_dict["criterion"]
#     device = args_dict["device"] if "device" in args_dict else "cuda"
#     device_ids = args_dict["device_ids"] if "device_ids" in args_dict else None
#     staging_device = args_dict["staging_device"] if "staging_device" in args_dict else None
#     symmetric_KL = lambda a, b : 0.5 * (criterion(F.log_softmax(a, dim=1), F.softmax(b, dim=1)) + criterion(F.log_softmax(b, dim=1), F.softmax(a, dim=1)))
#     dev_bsz = args_dict["dev_bsz"] if "dev_bsz" in args_dict else 64

#     if staging_device==None:
#         staging_device = f"cuda:{device_ids[0]}" if device_ids == None else "cuda"
#     results = {}
#     for phase in ['train', 'dev']:
#         if phase == 'train':
#             model.train()    # Set model to training mode
#         else:
#             dataloader[phase].set_bsz(dev_bsz)
#             model.eval()     # Set model to evaluate mode

#         running_loss = 0.0
#         tot_cnt = 0

#         # Iterate over data.
#         current_data = dataloader[phase].load_data()
#         while not current_data["reset"]:
#             input_embeddings, seq_len, input_masks, input_mask_invert, target_ids, target_mask, sentiment_labels, sent_level_EEG = current_data["data"]

#             input_embeddings_batch = input_embeddings
#             input_masks_batch = input_masks
#             input_mask_invert_batch = input_mask_invert
#             target_ids_batch = target_ids

#             """replace padding ids in target_ids with -100"""
#             target_ids_batch[target_ids_batch == tokenizer.pad_token_id] = -100

#             optimizer.zero_grad()

#             args_dict = {
#                 "input_data_batch" : input_embeddings_batch.to(staging_device, dtype=torch.float32),
#                 "input_masks_batch" : input_masks_batch.to(staging_device, dtype=torch.float32),
#                 "input_masks_invert" : input_mask_invert_batch.to(staging_device, dtype=torch.float32),
#                 "target_ids_batch" : target_ids_batch.to(staging_device),
#                 "pool_result" : False
#                 }

#             output = model(
#                 mode="PRETRAIN-EEG-TEXT-CLIP-MATCHING",
#                 args_dict=args_dict,
#                 staging_device=staging_device,
#             )

#             target_embed = current_data["target"].to(torch.float32)
#             target_embeds_pooled = torch.mean(target_embed, dim=1)
#             target_pairwise_embeds = torch.mm(target_embeds_pooled, target_embeds_pooled.T)

#             output = torch.mean(output, dim=1).to(torch.float32)
#             output = torch.mm(output, target_embeds_pooled.T)

#             loss = symmetric_KL(output, target_pairwise_embeds)

#             # Backward + Optimize only if in training phase
#             if phase == 'train':
#                 if device_ids == None:
#                     loss.backward()
#                     optimizer.step()
#                 else:
#                     loss.mean().backward()
#                     optimizer.step()

#             # Compute stats
#             if device_ids == None or len(device_ids) == 1:
#                 running_loss += loss.item() * input_embeddings_batch.size()[0]
#             else:
#                 running_loss += loss.mean().item() * input_embeddings_batch.size()[0]
#             tot_cnt += input_embeddings_batch.size()[0]
#             current_data = dataloader[phase].load_data()

#         epoch_loss = running_loss / tot_cnt

#         results[f"{phase}_loss"] = epoch_loss
#     results["model"] = model
#     return results

## Code2

In [5]:
from tqdm import tqdm

num_epochs = 50

In [6]:
from transformers import CLIPTokenizer
model = model
dataloader = dataset_dict["ZuCo-CLIP"]
optimizer = optim.Adam(model.parameters(), lr=1e-5) # LR has to be small, or else gradients will explode
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")
staging_device = "cuda:0"
criterion = nn.KLDivLoss(reduction="batchmean")
# symmetric_KL = lambda a, b : 0.5 * (criterion(F.log_softmax(a, dim=1), F.softmax(b, dim=1)) + criterion(F.log_softmax(b, dim=1), F.softmax(a, dim=1)))

In [7]:
import PRETRAIN_EEG_TEXT_CLIP_MATCHING

In [8]:
for epoch_num in tqdm(range(num_epochs)):
    args_dict = {
        "model" : model,
        "dataloader" : dataloader,
        "optimizer" : optimizer,
        "criterion" : criterion,
        "tokenizer" : tokenizer,
        "device" : "cuda",
        "device_ids" : None,
        "staging_device" : staging_device,
    }
    results = PRETRAIN_EEG_TEXT_CLIP_MATCHING.train(args_dict)
    model = results["model"]
    print(f"Epoch {epoch_num} train_loss: {results['train_loss']} dev_loss: {results['dev_loss']}")

  2%|▏         | 1/50 [00:28<23:31, 28.80s/it]

Epoch 0 train_loss: 140.18556053481416 dev_loss: 143.41725387573243


  4%|▍         | 2/50 [00:56<22:24, 28.02s/it]

Epoch 1 train_loss: 134.5408447909068 dev_loss: 141.03394438091078


  6%|▌         | 3/50 [01:23<21:46, 27.80s/it]

...

Epoch 47 train_loss: 106.46265085059476 dev_loss: 136.22547872442948


 98%|█████████▊| 49/50 [22:29<00:27, 27.62s/it]

Epoch 48 train_loss: 105.02776529702795 dev_loss: 141.34206510844984


100%|██████████| 50/50 [22:57<00:00, 27.55s/it]

Epoch 49 train_loss: 104.27238914765508 dev_loss: 145.53105002955385


In [8]:
import numpy as np

def evaluate(args_dict):
    dataloader = args_dict["dataloader"]
    model = args_dict["model"]
    tokenizer = args_dict["tokenizer"]
    device = args_dict["device"] if "device" in args_dict else "cuda"
    device_ids = args_dict["device_ids"] if "device_ids" in args_dict else None
    staging_device = args_dict["staging_device"] if "staging_device" in args_dict else None
    dev_bsz = args_dict["dev_bsz"] if "dev_bsz" in args_dict else 64

    if staging_device==None:
        staging_device = f"cuda:{device_ids[0]}" if device_ids == None else "cuda"
    results = {}
    for phase in ['train', 'dev']:
        if phase == 'train':
            model.train()    # Set model to training mode
        else:
            dataloader[phase].set_bsz(dev_bsz)
            model.eval()     # Set model to evaluate mode

        correct = 0
        tot_cnt = 0

        # Iterate over data.
        current_data = dataloader[phase].load_data()
        while not current_data["reset"]:
            input_embeddings, seq_len, input_masks, input_mask_invert, target_ids, target_mask, sentiment_labels, sent_level_EEG = current_data["data"]

            input_embeddings_batch = input_embeddings
            input_masks_batch = input_masks
            input_mask_invert_batch = input_mask_invert
            target_ids_batch = target_ids

            """replace padding ids in target_ids with -100"""
            target_ids_batch[target_ids_batch == tokenizer.pad_token_id] = -100

            model.zero_grad()

            args_dict = {
                "input_data_batch" : input_embeddings_batch.to(staging_device, dtype=torch.float32),
                "input_masks_batch" : input_masks_batch.to(staging_device, dtype=torch.float32),
                "input_masks_invert" : input_mask_invert_batch.to(staging_device, dtype=torch.float32),
                "target_ids_batch" : target_ids_batch.to(staging_device),
                "pool_result" : False
                }

            output = model(
                mode="PRETRAIN-EEG-TEXT-CLIP-MATCHING",
                args_dict=args_dict,
                staging_device=staging_device,
            )

            target_embed = current_data["target"].to(torch.float32)
            target_embeds_pooled = torch.mean(target_embed, dim=1)
            target_pairwise_embeds = torch.mm(target_embeds_pooled, target_embeds_pooled.T)

            output = torch.mean(output, dim=1).to(torch.float32)
            output = torch.mm(output, target_embeds_pooled.T)

            for i in range(output.shape[0]):
                current = torch.argmax(output[i])
                if current == i:
                    correct += 1
                tot_cnt += 1

            current_data = dataloader[phase].load_data()
            results[f"{phase}_accuracy"] = correct / tot_cnt
    return results

In [9]:
args_dict = {
    "model" : model,
    "dataloader" : dataloader,
    "tokenizer" : tokenizer,
    "device" : "cuda",
    "device_ids" : None,
    "staging_device" : staging_device,
}
results = evaluate(args_dict)

In [10]:
results

{'train_accuracy': 0.01721556886227545, 'dev_accuracy': 0.01953125}